# Create Russian Science Foundation Awards (GRANT PATTERN, project-card scrape)

The Russian Science Foundation (Российский научный фонд, RSF/РНФ) is Russia's
principal competitive science funder (founded 2013; absorbed RFBR's programs
in 2022). RSF publishes a public **project card** per grant at
`rscf.ru/project/{grant_number}/` with title, PI, host organization + region,
competition, research-area classifier, keywords, GRNTI code, and abstract —
all in Russian (kept verbatim; RSF publishes no English card fields).

The source script `scripts/local/rscf_to_s3.py` fetches these cards. RSF's
public project **search backend is broken** (the Bitrix `ext-filter` XHR
target 404s and the internal system `grant.rscf.ru` is login-gated), and no
sitemap lists card URLs, so there is **no public enumeration** of grant
numbers. The harvest is therefore **seeded from the RSF grant numbers
OpenAlex already carries** (funder F4320324099 `/awards` rows from
Crossref/text-mined sources, cleaned to the canonical `YY-NN-NNNNN` shape,
~15.7k unique). This ingest enriches those stubs with the funder's own
authoritative card data; it cannot discover RSF grants that funded zero
OpenAlex-indexed works (documented limitation — re-seed when RSF fixes its
search).

**Awarding body:** Russian Science Foundation — F4320324099 (RU). Path A
(F4320* Crossref-registered funder, present in `openalex.common.funder`);
canonical ror_id/doi come from the dim in Step 1.6.

**Schema choices / known limitations:**
- `funder_award_id` = the native RSF grant number (e.g. `22-29-01013`).
- **NO amounts anywhere**: RSF project cards publish no per-project funding
  amount -> `amount`/`currency` NULL on every row. **Step 6.7 amount check
  WAIVED** (source publishes no amounts).
- **Dates**: cards publish no start/end dates. `start_year` = 2000 + the
  grant-number `YY` prefix (validated 13-30). `start_date`/`end_date`/
  `end_year` NULL (no false precision). The competition year (often
  `start_year - 1`) is kept in the raw table's `competition_year`.
- **PI**: Russian name order **Family Given Patronymic** — the script splits
  family = first token, given = remaining tokens (NOT the Western
  `split_name`); academic degree stripped into its own raw column. No ORCIDs
  published -> `orcid` NULL. `affiliation.name` = the funding organization
  (Russian legal name), `country = 'RU'`.
- `funder_scheme` = the competition name (e.g. «Проведение фундаментальных
  научных исследований ... малыми отдельными научными группами»);
  `funding_type = 'research'` for all rows (RSF project competitions).
- `display_name`/`description` are the Russian title/abstract, verbatim.

**Prerequisites:** run `scripts/local/rscf_to_s3.py` first (checkpointed,
resumable; ~15.7k card fetches) to build + upload the parquet.

**Data source:** https://rscf.ru/project/ (project cards)
**S3 location:** `s3a://openalex-ingest/awards/rscf/rscf_projects.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rscf_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rscf/rscf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.rscf_raw;

## Step 1.5: Inspect raw + money/currency scan

Per runbook §1.5, scan every column for money-shaped data even though the
source is known to publish none. Expected result: **zero money-flavored
columns** — RSF project cards carry no amounts (see header; §6.7 waived).

In [ ]:
%sql
DESCRIBE openalex.awards.rscf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rscf_raw LIMIT 5;

In [ ]:
%sql
-- Money/currency-flavored column scan (runbook §1.5). Expect 0 rows both.
SELECT column_name FROM (DESCRIBE openalex.awards.rscf_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|currenc|ccy|iso_4217';

## Step 1.6: Fail-fast — verify RSF funder row exists (Path A, F4320*)

The Step 2 transform `CROSS JOIN`s against this lookup; if it returns 0 rows
the output table is silently empty. RSF is F4320324099 (Crossref-registered),
so the dim MUST return exactly 1 row — if it doesn't, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320324099;  -- Russian Science Foundation (expect exactly 1 row)

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rscf_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320324099  -- Russian Science Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(r.funder_award_id)
    ))) % 9000000000 AS id,
    r.display_name,                                 -- Russian project title (verbatim)
    r.description,                                  -- Russian abstract (verbatim)
    f.funder_id,
    r.funder_award_id,                              -- native RSF grant number
    CAST(NULL AS DOUBLE) AS amount,                 -- RSF publishes no amounts (§6.7 waived)
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,                     -- RSF project competitions
    r.funder_scheme,                                -- competition name (Russian)
    'rscf' AS provenance,
    CAST(NULL AS DATE) AS start_date,               -- cards publish no dates
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(r.start_year AS INT) AS start_year,    -- 2000 + grant-number YY prefix
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN r.lead_family_name IS NULL OR r.lead_family_name = '' THEN NULL
        ELSE struct(
            NULLIF(TRIM(r.lead_given_name), '') AS given_name,
            TRIM(r.lead_family_name) AS family_name,
            CAST(NULL AS STRING) AS orcid,          -- not published
            CAST(NULL AS DATE) AS role_start,
            struct(
                NULLIF(TRIM(r.organization), '') AS name,
                'RU' AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) AS investigators,
    r.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(r.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.rscf_raw r
CROSS JOIN funder_resolved f
WHERE r.funder_award_id IS NOT NULL
  AND r.display_name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 399

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rscf' AND priority = 399;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    399 as priority  -- RSF priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.rscf_awards;

## Step 6: Verification

Full §6.1–6.8. **§6.7 amount check is WAIVED**: RSF project cards publish no
per-project amounts (see header) — expect 0% amount coverage by design.
Everything else follows the standard gates (>90% titles, PI long-tail, etc.).

In [ ]:
%sql
SELECT COUNT(*) AS total_rscf_award_rows FROM openalex.awards.rscf_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.rscf_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_year) AS has_start_year,
    COUNT(lead_investigator) AS has_pi,
    COUNT(lead_investigator.affiliation.name) AS has_org,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) AS pct_pi,
    ROUND(COUNT(start_year) * 100.0 / COUNT(*), 1) AS pct_start_year
FROM openalex.awards.rscf_awards;

In [ ]:
%sql
-- §6.7 amount check — WAIVED: RSF publishes no per-project amounts.
-- Expect has_amount = 0 and distinct_currencies = 0 BY DESIGN.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.rscf_awards;

In [ ]:
%sql
-- §6.4 sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title, funder_award_id,
       lead_investigator.given_name, lead_investigator.family_name,
       SUBSTRING(lead_investigator.affiliation.name, 1, 50) AS org,
       start_year, SUBSTRING(funder_scheme, 1, 50) AS scheme
FROM openalex.awards.rscf_awards LIMIT 10;

In [ ]:
%sql
-- §6.4a PI frequency check — top names must be a real long-tail (PI re-grants
-- exist: RSF PIs win multiple competitions, so counts of 2-8 are normal;
-- hundreds = scraper bug). Names are Russian, family = first card token.
SELECT lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       COUNT(*) AS n
FROM openalex.awards.rscf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a display_name frequency — repeated identical titles = template bug
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.rscf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.6 year distribution — expect 2014-2025 (RSF founded 2013)
SELECT start_year, COUNT(*) AS cnt
FROM openalex.awards.rscf_awards
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- §6.5 funder consistency — should be exactly Russian Science Foundation
SELECT funder.id, funder.display_name, funder_id, COUNT(*) AS n
FROM openalex.awards.rscf_awards
GROUP BY 1, 2, 3;

In [ ]:
%sql
-- §6.8 confirm rows reached the shared raw table at the right priority
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rscf'
GROUP BY provenance, priority;